In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import os
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore")

In [16]:
# ========================= CONFIG =========================
CHANNELS = list(range(41, 47))                    # channels 41-46
CONTAMINATION_VALUES = [0.005, 0.01, 0.015, 0.02, 0.03, 0.05]   # various values to try
DECISION_THRESHOLDS = [0.0, -0.05, -0.1, -0.15]  # lower = more anomalies
RANDOM_SEED = 42
N_ESTIMATORS = 200
MAX_SAMPLES = 256

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)

print("=== ESA-ADB Baseline: Isolation Forest on channels 41-46 ===\n")
print(f"Testing {len(CONTAMINATION_VALUES)} contamination values × {len(DECISION_THRESHOLDS)} thresholds\n")

=== ESA-ADB Baseline: Isolation Forest on channels 41-46 ===

Testing 6 contamination values × 4 thresholds



In [17]:
# ========================= 1. PREPROCESS TRAIN (cached once) =========================
cache_file = os.path.join(CACHE_DIR, "preprocessed_train_channels41-46.pkl")

if os.path.exists(cache_file):
    print("✅ Loading cached train data...")
    data_scaled, df_train, _ = joblib.load(cache_file)
else:
    print("Preprocessing train.parquet...")
    df_train = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/train.parquet")

    channel_cols = [f'channel_{i}' for i in CHANNELS]
    df_train = df_train[['id'] + channel_cols + ['is_anomaly']].copy()
    df_train = df_train.sort_values('id').reset_index(drop=True)

    df_train[channel_cols] = df_train[channel_cols].ffill().bfill()

    data_scaled = df_train[channel_cols].values.copy()   # No scaling for Isolation Forest

    joblib.dump((data_scaled, df_train, None), cache_file)
    print(f"✅ Train preprocessing cached. Shape: {data_scaled.shape}")

true_labels = df_train['is_anomaly'].values
print(f"True anomaly rate in train: {true_labels.mean():.4%}\n")

✅ Loading cached train data...
True anomaly rate in train: 10.4839%



In [ ]:
# ========================= 2. LOOP OVER CONTAMINATION + THRESHOLD =========================
results = []

for cont in CONTAMINATION_VALUES:
    model_file = os.path.join(CACHE_DIR, f"isolation_forest_cont{cont:.4f}.pkl")

    # Train or load model for this contamination
    if os.path.exists(model_file):
        print(f"Loading model (contamination={cont}) from cache...")
        model = joblib.load(model_file)
    else:
        print(f"Training model (contamination={cont})...")
        model = IsolationForest(
            n_estimators=N_ESTIMATORS,
            max_samples=MAX_SAMPLES,
            contamination=cont,
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
        model.fit(data_scaled)
        joblib.dump(model, model_file)
        print("   → Model cached")

    # Get decision scores once
    scores = model.decision_function(data_scaled)

    # Try different thresholds
    for thresh in DECISION_THRESHOLDS:
        pred = (scores < thresh).astype(int)

        prec = precision_score(true_labels, pred, zero_division=0)
        rec  = recall_score(true_labels, pred, zero_division=0)
        f1   = f1_score(true_labels, pred, zero_division=0)

        results.append({
            'contamination': cont,
            'threshold': thresh,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'pred_rate': pred.mean()
        })

        print(f"   cont={cont:.4f} | thresh={thresh:6} → Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f} | Pred rate={pred.mean():.4%}")

Training model (contamination=0.005)...
   → Model cached
   cont=0.0050 | thresh=   0.0 → Prec=1.0000 Rec=0.0476 F1=0.0910 | Pred rate=0.4996%
   cont=0.0050 | thresh= -0.05 → Prec=1.0000 Rec=0.0317 F1=0.0614 | Pred rate=0.3322%
   cont=0.0050 | thresh=  -0.1 → Prec=1.0000 Rec=0.0088 F1=0.0174 | Pred rate=0.0921%
   cont=0.0050 | thresh= -0.15 → Prec=0.0000 Rec=0.0000 F1=0.0000 | Pred rate=0.0000%
Training model (contamination=0.01)...
   → Model cached
   cont=0.0100 | thresh=   0.0 → Prec=1.0000 Rec=0.0954 F1=0.1741 | Pred rate=0.9999%
   cont=0.0100 | thresh= -0.05 → Prec=1.0000 Rec=0.0760 F1=0.1413 | Pred rate=0.7968%
   cont=0.0100 | thresh=  -0.1 → Prec=1.0000 Rec=0.0390 F1=0.0751 | Pred rate=0.4092%
   cont=0.0100 | thresh= -0.15 → Prec=1.0000 Rec=0.0297 F1=0.0576 | Pred rate=0.3111%
Training model (contamination=0.015)...


In [ ]:
# ========================= 3. SUMMARY TABLE =========================
print("\n" + "="*80)
print("SUMMARY OF ALL COMBINATIONS (local proxy)")
print("="*80)
summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values('f1', ascending=False).reset_index(drop=True)
print(summary_df.round(4))

# Best combination (by proxy F1)
best = summary_df.iloc[0]
print(f"\nBest local proxy: contamination={best['contamination']}, threshold={best['threshold']} (F1 = {best['f1']:.4f})")


=== Local Proxy Score on Train Set ===
Precision : 1.0000
Recall    : 0.0476
F1-score  : 0.0910
Predicted anomaly rate: 0.4996% | True rate: 10.4839%


In [ ]:
# ========================= 4. GENERATE SUBMISSION WITH BEST COMBINATION =========================
print("\nGenerating final submission using best combination...")

best_cont = best['contamination']
best_thresh = best['threshold']

# Load the best model
model_file = os.path.join(CACHE_DIR, f"isolation_forest_cont{best_cont:.4f}.pkl")
model = joblib.load(model_file)

# Load test
df_test = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/test.parquet")
channel_cols = [f'channel_{i}' for i in CHANNELS]
df_test = df_test[['id'] + channel_cols].copy()
df_test = df_test.sort_values('id').reset_index(drop=True)
df_test[channel_cols] = df_test[channel_cols].ffill().bfill()

test_data = df_test[channel_cols].values
test_scores = model.decision_function(test_data)
test_pred = (test_scores < best_thresh).astype(int)

submission = pd.DataFrame({
    'id': df_test['id'],
    'anomaly_flag': test_pred
})

submission.to_parquet("submission_isoforest_best_channels41-46.parquet", index=False)

print(f"✅ Submission saved as 'submission_isoforest_best_channels41-46.parquet'")
print(f"   → Used contamination = {best_cont}, threshold = {best_thresh}")
print(f"   → Predicted anomalies in test: {test_pred.sum()} ({test_pred.mean():.4%})")

print("\nYou can now upload this file as Late Submission on Kaggle to get the real corrected event-wise F0.5 score.")


Loading test.parquet and generating submission...
Submission saved! Shape: (521280, 2)
Predicted anomalies in test: 0 (0.0000%)

First 5 rows of submission:
         id  anomaly_flag
0  14728321             0
1  14728322             0
2  14728323             0
3  14728324             0
4  14728325             0

✅ Done! You can now upload 'submission_isoforest_channels41-46.parquet' as Late Submission on Kaggle to get the real score.
